# Modul 2 - Indexing: Incidence Matrix dan Inverted Index
**Praktikum Step 2 (Indexing), STKI**

| Keterangan | Isi |
|---|---|
| Data | 30 dokumen (D01 sampai D30) dari `data/korpus.csv` |
| Alat | Python, pandas, Sastrawi |

## Cara menjalankan
1. Di panel file Colab (ikon folder di kiri), buat folder bernama `data`.
2. Unggah `korpus.csv` ke dalam folder `data`.
3. Jalankan sel secara berurutan dari atas ke bawah. Sel persiapan harus dijalankan lebih dulu.

## Pendahuluan
Modul ini membangun dua komponen indexing: *incidence matrix* dan *inverted index*. Index yang dibangun juga dipakai untuk *positional index*, *phrase query*, dan *proximity search*. Pemrosesan teks memakai fungsi `preprocess()` yang sama dengan Modul 1. Sel persiapan di bawah menyalinnya agar notebook ini dapat berjalan sendiri.

In [ ]:
!pip install Sastrawi -q

import re
import json
import os
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

# ---------------------------------------------------------------------------
# Persiapan: memuat korpus dan fungsi preprocess() (sama dengan Modul 1)
# ---------------------------------------------------------------------------
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

korpus = pd.read_csv(Path("data/korpus.csv"))
korpus["teks"] = korpus["judul"] + ". " + korpus["isi"]

def corpus_texts():
    return dict(zip(korpus["doc_id"], korpus["teks"]))

def case_folding(teks):
    return teks.lower()

def tokenize(teks):
    return re.findall(r"[a-z0-9]+", teks.lower())

STOPWORDS = set(StopWordRemoverFactory().get_stop_words())
stemmer = StemmerFactory().create_stemmer()
_cache = {}

def stem(token):
    if token not in _cache:
        _cache[token] = stemmer.stem(token)
    return _cache[token]

def remove_stopwords(tokens, stopwords=STOPWORDS):
    return [t for t in tokens if t not in stopwords]

def preprocess(teks, use_stopword=True, use_stemming=True):
    tokens = tokenize(case_folding(teks))
    if use_stopword:
        tokens = remove_stopwords(tokens)
    if use_stemming:
        tokens = [stem(t) for t in tokens]
    return tokens

teks = corpus_texts()
docs = {d: preprocess(t) for d, t in teks.items()}
print("[Persiapan] Memuat dokumen")
print("Jumlah dokumen:", len(docs))
print("D07 setelah preprocess:", docs["D07"])

## Langkah 1 - Term-document incidence matrix
Matriks biner berukuran |V| x N. Sel bernilai 1 jika term muncul di dokumen tersebut.

In [ ]:
print("[Langkah 1] Term-document incidence matrix")
subset = ["D07", "D08", "D09", "D10", "D11", "D12"]
vocab_subset = sorted({t for d in subset for t in docs[d]})
incidence = pd.DataFrame([[int(t in docs[d]) for d in subset] for t in vocab_subset],
                         index=vocab_subset, columns=subset)
print("Ukuran matriks:", incidence.shape)
print(incidence.loc[["air", "hama", "padi", "pupuk", "sawah", "tani", "tanah"]].to_string())

# Kepadatan matriks pada seluruh korpus
vocab = sorted({t for tokens in docs.values() for t in tokens})
sel_total = len(vocab) * len(docs)
sel_isi = sum(len(set(tokens)) for tokens in docs.values())
print(f"\nKosakata: {len(vocab)} term, {len(docs)} dokumen")
print(f"Sel matriks penuh : {sel_total}")
print(f"Sel bernilai 1    : {sel_isi} ({sel_isi / sel_total:.1%})")

## Langkah 2 - Inverted index
Struktur: `index[term] = {doc_id: tf}`. Panjang posting list sama dengan df.

In [ ]:
print("[Langkah 2] Inverted index")

def build_inverted_index(docs_tokens):
    index = defaultdict(dict)
    for doc_id, tokens in docs_tokens.items():
        for term, tf in Counter(tokens).items():
            index[term][doc_id] = tf
    return dict(sorted(index.items()))

index = build_inverted_index(docs)
print("Jumlah term di kamus:", len(index))
for term in ["tani", "digital", "gizi", "cari"]:
    print(f"{term:8s} df={len(index[term]):2d}  postings={index[term]}")

# Tabel df dan cf (total tf) untuk term paling sering muncul di dokumen
tabel = pd.DataFrame({
    "df": {t: len(p) for t, p in index.items()},
    "cf (total tf)": {t: sum(p.values()) for t, p in index.items()},
}).sort_values("df", ascending=False)
print("\n10 term dengan df tertinggi:")
print(tabel.head(10).to_string())

## Langkah 3 - Menyimpan dan memuat index

In [ ]:
print("[Langkah 3] Menyimpan dan memuat index")
os.makedirs("output", exist_ok=True)
with open("output/inverted_index.json", "w") as f:
    json.dump(index, f)

with open("output/inverted_index.json") as f:
    index_dimuat = json.load(f)
print("Index tersimpan dan dimuat ulang dengan benar:", index_dimuat == index)
print("Ukuran file:", os.path.getsize("output/inverted_index.json"), "byte")

## Langkah 4 - Positional index dan phrase query
Positional index menyimpan posisi kemunculan term: `index[term][doc_id] = [posisi, ...]`.

In [ ]:
print("[Langkah 4] Positional index dan phrase query")

def build_positional_index(docs_tokens):
    index = defaultdict(lambda: defaultdict(list))
    for doc_id, tokens in docs_tokens.items():
        for pos, term in enumerate(tokens):
            index[term][doc_id].append(pos)
    return {t: dict(p) for t, p in sorted(index.items())}

pos_index = build_positional_index(docs)
print("cerdas :", pos_index["cerdas"])
print("buat   :", pos_index["buat"])

def phrase_search(frasa, pos_index):
    terms = preprocess(frasa)
    if not terms or any(t not in pos_index for t in terms):
        return []
    kandidat = set.intersection(*(set(pos_index[t]) for t in terms))
    hasil = []
    for d in sorted(kandidat):
        for p in pos_index[terms[0]][d]:
            if all(p + i in pos_index[t][d] for i, t in enumerate(terms[1:], start=1)):
                hasil.append(d)
                break
    return hasil

for frasa in ["kecerdasan buatan", "pertanian cerdas", "pupuk organik", "data pribadi"]:
    print(f"{frasa!r:22s} -> {phrase_search(frasa, pos_index)}")

---
# Tugas Step 2

## Tugas 1 - Perbandingan ukuran incidence matrix dan inverted index
Matriks menyimpan semua sel |V| x N. Inverted index hanya menyimpan pasangan term-dokumen yang bernilai tf > 0 (posting), ditambah daftar kamus.

In [ ]:
print("[Tugas 1] Perbandingan ukuran")
entri_matriks = len(index) * len(docs)
entri_posting = sum(len(p) for p in index.values())
entri_kamus = len(index)
print("Entri matriks penuh :", entri_matriks)
print("Entri posting       :", entri_posting)
print("Entri kamus         :", entri_kamus)
print(f"Rasio (posting saja): {entri_matriks / entri_posting:.2f} kali")
print(f"Rasio (+ kamus)     : {entri_matriks / (entri_posting + entri_kamus):.2f} kali")

### Jawaban Tugas 1
Matriks penuh memiliki 472 term x 30 dokumen = 14.160 sel. Inverted index menyimpan 702 posting, yaitu jumlah sel bernilai 1 pada matriks. Rasio posting terhadap sel matriks sekitar **20,2 kali**. Jika 472 entri kamus ikut dihitung, total entri index menjadi 1.174 dan rasionya sekitar **12,1 kali**.

Inverted index lebih hemat karena sel bernilai 0 (sekitar 95% dari matriks) tidak disimpan. Pada koleksi yang lebih besar, rasio ini semakin besar, karena matriks tumbuh dengan |V| x N, sedangkan jumlah posting hanya tumbuh sesuai jumlah kemunculan term.

## Tugas 2 - Term dengan df = 1 (hapax)
Hapax adalah term yang hanya muncul di satu dokumen.

In [ ]:
print("[Tugas 2] Hapax")
hapax = [t for t, p in index.items() if len(p) == 1]
print("Jumlah hapax:", len(hapax))
for t in hapax[:10]:
    d, tf = next(iter(index[t].items()))
    print(f"{t:15s} dokumen={d} tf={tf}")

### Jawaban Tugas 2
Pada korpus ini terdapat **346 hapax** dari 472 term (sekitar 73%). Sepuluh contohnya dicetak oleh kode di atas.

Hapax berguna untuk pencarian ketika kueri memuatnya. Term yang hanya ada di sedikit dokumen memiliki nilai IDF tinggi, sehingga kecocokan padanya sangat membedakan dokumen. Contohnya, "aedes" dan "aegypti" mengarah ke D01 (Demam Berdarah Dengue). Namun hapax tidak membantu pada kueri umum yang term-nya sudah tersebar di banyak dokumen.

Sebagian hapax tidak bermakna untuk pencarian, misalnya angka "150" (D03) dan "2" (D04, dari "tipe 2"). Sebagian lain dapat berupa salah ketik atau bentuk stemming yang tidak sempurna. Kode "3m" (D01, dari gerakan 3M) tetap berguna jika kueri memuat kode tersebut. Token seperti ini tidak diubah oleh stemming, sehingga tetap menjadi hapax.

Kesimpulannya, hapax berguna untuk kueri yang spesifik, tetapi perlu dinilai satu per satu.

## Tugas 3 - Posisi sebelum atau sesudah stopword dihapus
Index lama dibangun setelah stopword dihapus. Index baru menyimpan posisi asli dengan memanggil `preprocess(use_stopword=False)`, sehingga stopword tetap menempati posisi.

In [ ]:
print("[Tugas 3] Posisi sebelum stopword dihapus")
docs_raw = {d: preprocess(t, use_stopword=False) for d, t in teks.items()}
pos_index_raw = build_positional_index(docs_raw)

def phrase_search_raw(frasa, pos_index):
    terms = preprocess(frasa, use_stopword=False)
    if not terms or any(t not in pos_index for t in terms):
        return []
    kandidat = set.intersection(*(set(pos_index[t]) for t in terms))
    hasil = []
    for d in sorted(kandidat):
        for p in pos_index[terms[0]][d]:
            if all(p + i in pos_index[t][d] for i, t in enumerate(terms[1:], start=1)):
                hasil.append(d)
                break
    return hasil

for frasa in ["sistem temu kembali informasi", "gerakan 3M", "air pupuk"]:
    print(f"{frasa!r:34s} setelah stopword : {phrase_search(frasa, pos_index)}")
    print(f"{frasa!r:34s} sebelum stopword : {phrase_search_raw(frasa, pos_index_raw)}")

### Jawaban Tugas 3
Pada index yang dibangun dari teks setelah stopword dihapus, kata yang dipisahkan oleh stopword menjadi berurutan. D10 memuat "air dan pupuk". Setelah "dan" dihapus, index mencatat "air" dan "pupuk" pada posisi berurutan, sehingga phrase query "air pupuk" cocok dengan D10, padahal di teks asli kedua kata itu tidak berurutan. Ini adalah *false positive*. Versi index yang mempertahankan posisi asli tidak mengembalikan D10.

Dua frasa lain ("sistem temu kembali informasi" dan "gerakan 3M") tidak terpengaruh di korpus ini, karena di teks asli kedua frasa sudah berurutan tanpa stopword di antaranya (D29 dan D01). Hasil kedua versi index sama untuk keduanya.

| Frasa | Hasil index lama (setelah stopword) | Hasil index baru (sebelum stopword) |
|---|---|---|
| sistem temu kembali informasi | D29 | D29 |
| gerakan 3M | D01 | D01 |
| air pupuk | D10 | tidak ada |

Kesimpulannya, posisi harus dihitung dari teks asli agar jarak antarkata akurat.

## Tugas 4 - Proximity search
`proximity_search(t1, t2, k)` mengambil selisih posisi terkecil antara t1 dan t2 di setiap dokumen, lalu mengembalikan dokumen jika selisih itu tidak lebih dari k. Fungsi menerima `pos_index` sebagai argumen keempat, sehingga dapat dipakai dengan index mana pun.

In [ ]:
print("[Tugas 4] Proximity search")

def proximity_search(t1, t2, k, pos_index):
    a = preprocess(t1, use_stopword=False)
    b = preprocess(t2, use_stopword=False)
    if not a or not b or a[0] not in pos_index or b[0] not in pos_index:
        return []
    a, b = a[0], b[0]
    kandidat = set(pos_index[a]) & set(pos_index[b])
    hasil = []
    for d in sorted(kandidat):
        jarak = min((abs(p1 - p2) for p1 in pos_index[a][d] for p2 in pos_index[b][d]
                     if not (a == b and p1 == p2)), default=None)
        if jarak is not None and jarak <= k:
            hasil.append(d)
    return hasil

print("cerdas, buat, k=3   ->", proximity_search("cerdas", "buat", 3, pos_index_raw))
print("cerdas, buat, k=1   ->", proximity_search("cerdas", "buat", 1, pos_index_raw))
print("padi, hama, k=5     ->", proximity_search("padi", "hama", 5, pos_index_raw))

### Jawaban Tugas 4
Fungsi `proximity_search` mengambil selisih posisi terkecil antara term t1 dan t2 di setiap dokumen, lalu membandingkannya dengan k.

- `cerdas` dan `buat` dengan k=3 maupun k=1 sama-sama menghasilkan `['D14']`, karena di D14 kedua kata berjarak 1 posisi.
- `padi` dan `hama` dengan k=5 tidak menghasilkan dokumen. Di D07, "hama" berada di posisi 19 dan "padi" terdekat di posisi 26, jaraknya 7. Angka itu lebih besar dari 5, sehingga tidak ada dokumen yang lolos.

Kompleksitas per dokumen adalah O(|P1|·|P2|), yang cukup untuk korpus kecil. Pada indeks yang besar, dua posting list yang terurut dapat digabungkan dengan teknik two-pointer sehingga kompleksitasnya menjadi O(|P1|+|P2|). Phrase query merupakan kasus khusus proximity dengan urutan tetap dan jarak 1.

Hasil phrase query pada Langkah 4: "kecerdasan buatan" -> D14, "pertanian cerdas" -> D10, "pupuk organik" -> D08, "data pribadi" -> D15.

---
## Kesimpulan
- Indexing menghasilkan inverted index yang jauh lebih hemat dibanding incidence matrix (sekitar 12 hingga 20 kali, tergantung apakah kamus ikut dihitung).
- Inverted index memungkinkan phrase dan proximity search. Positional index harus dibangun dari posisi asli agar jarak antarkata akurat, seperti terlihat pada kasus "air pupuk" di D10.